# Data Check

Check which inference result files currently exist against all possible experiment combinations defined in `scripts/submit_job.sh`.

Covers:
- **Onepass** runs: summary keys in `output/{model}_onepass.json`
- **Two-pass** runs: summary keys in `output/{model}_twopass.json`
- **Nothink** runs: summary keys in `output/{model}_nothink.json` (uses `default` config profile)

In [13]:
"""All experiment combinations mirrored from scripts/submit_job.sh."""

import json
from pathlib import Path

# notebooks run from notebooks/, so step up to reach the repo root
OUTPUT_DIR = Path("..") / "output"

# models array from submit_job.sh
MODELS = [
    "qwen3-8b",
    "qwen3-14b",
    "qwen3-32b",
    "ocr-7b",
    "olmo-3-7b",
    "or-7b",
    "llama-r1-8b",
]

# code benchmarks only — eval_datasets array from submit_job.sh
DATASETS = [
    "code/evalplus",
    "code/livecodebench",
    "code/bigcodebench",
    "code/code_contests",
    "crux/cruxeval_i",
    "crux/cruxeval_o",
]

# config profile used for twopass runs
CONFIG = "greedy"

# config profiles to check for onepass and nothink runs
CONFIGS = [
    "greedy",
    # "default",
]

# total token budget — fixed across all runs
MAX_TOKENS = 32768

# max_think_tokens sweep values from submit_job.sh
MAX_THINK_TOKENS = [
    4096,
    8192,
    12288,
    16384,
    20480,
    24576,
    28762,
]

# overflow suffix keys from submit_job.sh
OVERFLOW_SUFFIXES = [
    "base",
    # "truncated",
    # "formal",
    # "human",
]

# these datasets require a separate execution-based evaluation pipeline;
# the summary records pass_at_1=0.0 as a placeholder until that is run
EXTERNALLY_EVALUATED = {"bigcodebench"}

In [14]:
"""Check all onepass result files and summary keys (model × dataset × config).

Checks both 'greedy' and 'default' config profiles. Individual result files live
at output/onepass/{model}/{stem}_{config}_mx{tokens}_onepass.json; summary keys
live in {model}_onepass.json.
"""

# column widths derived from the data
COL_D = max(len(d) for d in DATASETS) + 2
COL_S = 7  # matches "summary" header

# track totals for the summary at the end
onepass_found = 0
onepass_total = 0
onepass_evaluated = 0

for model in MODELS:
    # load the per-model onepass summary JSON once — shared across all configs
    summary_path = OUTPUT_DIR / f"{model}_onepass.json"
    summary = json.load(open(summary_path)) if summary_path.exists() else {}

    # collect results per config first so the overall model header can show totals
    config_rows: dict[str, list] = {}
    for config in CONFIGS:
        rows = []
        for dataset in DATASETS:
            stem = Path(dataset).stem
            file_path = (
                OUTPUT_DIR
                / "onepass"
                / model
                / f"{stem}_{config}_mx{MAX_TOKENS}_onepass.json"
            )
            file_exists = file_path.exists()
            summary_key = f"{dataset}_{config}_mx{MAX_TOKENS}_onepass"
            summary_exists = summary_key in summary

            # evaluated = summary exists; for externally-evaluated datasets only once
            # a real (non-zero) score has been computed
            if stem in EXTERNALLY_EVALUATED:
                evaluated = (
                    summary_exists and summary[summary_key].get("pass_at_1", 0) > 0
                )
            else:
                evaluated = summary_exists

            rows.append((dataset, file_exists, summary_exists, evaluated))
            onepass_total += 1
            onepass_found += int(file_exists)
            onepass_evaluated += int(evaluated)
        config_rows[config] = rows

    # overall model header: totals across all configs
    total = len(DATASETS) * len(CONFIGS)
    total_f = sum(int(f) for rows in config_rows.values() for _, f, _, _ in rows)
    total_s = sum(int(s) for rows in config_rows.values() for _, _, s, _ in rows)
    total_e = sum(int(e) for rows in config_rows.values() for _, _, _, e in rows)
    print(
        f"── {model} ── "
        f"(file: {total_f}/{total}, summary: {total_s}/{total}, evaluated: {total_e}/{total})"
    )

    for config, rows in config_rows.items():
        f_count = sum(int(f) for _, f, _, _ in rows)
        s_count = sum(int(s) for _, _, s, _ in rows)
        e_count = sum(int(e) for _, _, _, e in rows)
        n = len(DATASETS)
        print(
            f"  [{config}]  file: {f_count}/{n}, summary: {s_count}/{n}, evaluated: {e_count}/{n}"
        )
        print(
            f"  {'dataset':{COL_D}}  {'file':{COL_S}}  {'summary':{COL_S}}  evaluated"
        )
        print(f"  {'─' * COL_D}  {'─' * COL_S}  {'─' * COL_S}  {'─' * COL_S}")

        for dataset, file_exists, summary_exists, evaluated in rows:
            f_tick = "✓" if file_exists else "✗"
            s_tick = "✓" if summary_exists else "✗"
            stem = Path(dataset).stem
            if stem in EXTERNALLY_EVALUATED and not evaluated:
                e_tick = "ext"
            else:
                e_tick = "✓" if evaluated else "✗"
            print(f"  {dataset:{COL_D}}  {f_tick:{COL_S}}  {s_tick:{COL_S}}  {e_tick}")
        print()
    print()

── qwen3-8b ── (file: 6/6, summary: 6/6, evaluated: 6/6)
  [greedy]  file: 6/6, summary: 6/6, evaluated: 6/6
  dataset               file     summary  evaluated
  ────────────────────  ───────  ───────  ───────
  code/evalplus         ✓        ✓        ✓
  code/livecodebench    ✓        ✓        ✓
  code/bigcodebench     ✓        ✓        ✓
  code/code_contests    ✓        ✓        ✓
  crux/cruxeval_i       ✓        ✓        ✓
  crux/cruxeval_o       ✓        ✓        ✓


── qwen3-14b ── (file: 6/6, summary: 6/6, evaluated: 6/6)
  [greedy]  file: 6/6, summary: 6/6, evaluated: 6/6
  dataset               file     summary  evaluated
  ────────────────────  ───────  ───────  ───────
  code/evalplus         ✓        ✓        ✓
  code/livecodebench    ✓        ✓        ✓
  code/bigcodebench     ✓        ✓        ✓
  code/code_contests    ✓        ✓        ✓
  crux/cruxeval_i       ✓        ✓        ✓
  crux/cruxeval_o       ✓        ✓        ✓


── qwen3-32b ── (file: 6/6, summary: 6/6, eva

In [15]:
"""Check all two-pass summary keys (model × dataset × max_think_tokens × overflow_suffix).

Two-pass results are stored in per-model summary JSONs only ({model}_twopass.json);
there are no individual result files to check.
"""

# column widths derived from the data
COL_R = max(len(f"th{t}_{s}") for t in MAX_THINK_TOKENS for s in OVERFLOW_SUFFIXES) + 2
COL_S = max(len("partial (10/10)"), len("complete")) + 1


# helper: compact status label
def _status(n: int, total: int) -> str:
    if n == total:
        return "complete"
    if n == 0:
        return "missing"
    return f"partial ({n}/{total})"


# track totals for the summary at the end
twopass_found = 0
twopass_total = 0

model_combos = len(DATASETS) * len(MAX_THINK_TOKENS) * len(OVERFLOW_SUFFIXES)

for model in MODELS:
    # load the per-model twopass summary JSON once
    # keys: "{dataset}_{config}_mx{max_tokens}_th{max_think_tokens}_{suffix}"
    summary_path = OUTPUT_DIR / f"{model}_twopass.json"
    summary = json.load(open(summary_path)) if summary_path.exists() else {}

    # collect all run results for this model first so the counts can go in the header
    model_runs = []
    model_found = 0

    for tokens in MAX_THINK_TOKENS:
        for suffix in OVERFLOW_SUFFIXES:
            # run name encodes the think-token cap and suffix key
            run_name = f"th{tokens}_{suffix}"

            n_summary = 0
            missing_summaries = []

            for dataset in DATASETS:
                summary_key = f"{dataset}_{CONFIG}_mx{MAX_TOKENS}_{run_name}"
                summary_exists = summary_key in summary

                n_summary += int(summary_exists)
                if not summary_exists:
                    missing_summaries.append(dataset)

            model_found += n_summary
            model_runs.append((run_name, n_summary, missing_summaries))

    print(f"── {model} ── (summary: {model_found}/{model_combos})")
    print(f"  {'run':{COL_R}}  summary")
    print(f"  {'─' * COL_R}  {'─' * COL_S}")

    n = len(DATASETS)
    for run_name, n_summary, missing_summaries in model_runs:
        print(f"  {run_name:{COL_R}}  {_status(n_summary, n)}")

        # show missing summary datasets when partial (not fully missing)
        if 0 < n_summary < n:
            print(f"    Missing: {', '.join(missing_summaries)}")

    twopass_total += model_combos
    twopass_found += model_found
    print()

── qwen3-8b ── (summary: 42/42)
  run             summary
  ──────────────  ────────────────
  th4096_base     complete
  th8192_base     complete
  th12288_base    complete
  th16384_base    complete
  th20480_base    complete
  th24576_base    complete
  th28762_base    complete

── qwen3-14b ── (summary: 36/42)
  run             summary
  ──────────────  ────────────────
  th4096_base     complete
  th8192_base     complete
  th12288_base    complete
  th16384_base    complete
  th20480_base    complete
  th24576_base    missing
  th28762_base    complete

── qwen3-32b ── (summary: 42/42)
  run             summary
  ──────────────  ────────────────
  th4096_base     complete
  th8192_base     complete
  th12288_base    complete
  th16384_base    complete
  th20480_base    complete
  th24576_base    complete
  th28762_base    complete

── ocr-7b ── (summary: 42/42)
  run             summary
  ──────────────  ────────────────
  th4096_base     complete
  th8192_base     complete
  th1

In [16]:
"""Check all nothink summary keys (model × dataset × config).

Nothink runs inject an empty <think></think> block to skip reasoning entirely.
Results are stored in per-model summary JSONs only ({model}_nothink.json).
Checks both 'greedy' and 'default' config profiles.
"""

# column width reused from the onepass cell
COL_D = max(len(d) for d in DATASETS) + 2

# track totals for the summary at the end
nothink_found = 0
nothink_total = 0

for model in MODELS:
    # load the per-model nothink summary JSON once — shared across all configs
    summary_path = OUTPUT_DIR / f"{model}_nothink.json"
    summary = json.load(open(summary_path)) if summary_path.exists() else {}

    # collect results per config
    config_rows: dict[str, list] = {}
    for config in CONFIGS:
        rows = []
        for dataset in DATASETS:
            # key format: "{dataset}_{config}_mx{MAX_TOKENS}_nothink"
            summary_key = f"{dataset}_{config}_mx{MAX_TOKENS}_nothink"
            summary_exists = summary_key in summary
            rows.append((dataset, summary_exists))
            nothink_total += 1
            nothink_found += int(summary_exists)
        config_rows[config] = rows

    # overall model header: totals across all configs
    total = len(DATASETS) * len(CONFIGS)
    total_s = sum(int(s) for rows in config_rows.values() for _, s in rows)
    print(f"── {model} ── (summary: {total_s}/{total})")

    for config, rows in config_rows.items():
        s_count = sum(int(s) for _, s in rows)
        n = len(DATASETS)
        print(f"  [{config}]  summary: {s_count}/{n}")
        print(f"  {'dataset':{COL_D}}  summary")
        print(f"  {'─' * COL_D}  {'─' * 7}")

        for dataset, summary_exists in rows:
            s_tick = "✓" if summary_exists else "✗"
            print(f"  {dataset:{COL_D}}  {s_tick}")
        print()
    print()

── qwen3-8b ── (summary: 6/6)
  [greedy]  summary: 6/6
  dataset               summary
  ────────────────────  ───────
  code/evalplus         ✓
  code/livecodebench    ✓
  code/bigcodebench     ✓
  code/code_contests    ✓
  crux/cruxeval_i       ✓
  crux/cruxeval_o       ✓


── qwen3-14b ── (summary: 6/6)
  [greedy]  summary: 6/6
  dataset               summary
  ────────────────────  ───────
  code/evalplus         ✓
  code/livecodebench    ✓
  code/bigcodebench     ✓
  code/code_contests    ✓
  crux/cruxeval_i       ✓
  crux/cruxeval_o       ✓


── qwen3-32b ── (summary: 6/6)
  [greedy]  summary: 6/6
  dataset               summary
  ────────────────────  ───────
  code/evalplus         ✓
  code/livecodebench    ✓
  code/bigcodebench     ✓
  code/code_contests    ✓
  crux/cruxeval_i       ✓
  crux/cruxeval_o       ✓


── ocr-7b ── (summary: 6/6)
  [greedy]  summary: 6/6
  dataset               summary
  ────────────────────  ───────
  code/evalplus         ✓
  code/livecodebench    

In [17]:
"""Overall summary across all experiment types."""

total_found = onepass_found + twopass_found + nothink_found
total_all = onepass_total + twopass_total + nothink_total

print("══════════════════════════════")
print("  OVERALL SUMMARY")
print("══════════════════════════════")
print(
    f"  Onepass    {onepass_found:4d} / {onepass_total:4d}  ({onepass_found / onepass_total:.0%})"
    f"  evaluated: {onepass_evaluated}/{onepass_total}"
)
print(
    f"  Two-pass   {twopass_found:4d} / {twopass_total:4d}  ({twopass_found / twopass_total:.0%})"
)
print(
    f"  Nothink    {nothink_found:4d} / {nothink_total:4d}  ({nothink_found / nothink_total:.0%})"
)
print(
    f"  Total      {total_found:4d} / {total_all:4d}  ({total_found / total_all:.0%})"
)
print("══════════════════════════════")

══════════════════════════════
  OVERALL SUMMARY
══════════════════════════════
  Onepass      42 /   42  (100%)  evaluated: 42/42
  Two-pass    288 /  294  (98%)
  Nothink      42 /   42  (100%)
  Total       372 /  378  (98%)
══════════════════════════════
